# PA1 supplemental stabilization: unattended overnight run

This notebook trains and evaluates all requested supplemental variants while preserving the original results.

Before choosing **Save Version → Save & Run All**, enable a GPU and attach these datasets:

- `/kaggle/input/datasets/ahmadsarfraz345/local-cifar`
- `/kaggle/input/datasets/ahmadsarfraz345/local-cifar100`
- `/kaggle/input/datasets/ahmadsarfraz345/task2-4-checkpoints`

PACS is downloaded by the notebook. The notebook selects checkpoints and decides whether to run normalized-MMD fallbacks using source validation only. Sketch labels and CIFAR-100 are opened only after the corresponding selection manifests have been written.

At completion, download the ZIP files shown by the last cell from the saved notebook version's **Output** tab. Logs and partial results are archived even if an independent task fails.


In [ ]:
# Fresh repository setup. Push the supplemental implementation before running this notebook.
%cd /kaggle/working
from pathlib import Path
import shutil
checkout = Path('/kaggle/working/pa1-beyond-iid')
if checkout.exists():
    shutil.rmtree(checkout)
!git clone https://github.com/ahmad-sarfraz345/pa1-beyond-iid.git /kaggle/working/pa1-beyond-iid
%cd /kaggle/working/pa1-beyond-iid
!python -m pip install -q -r requirements.txt gdown


In [ ]:
# Environment and implementation preflight.
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import traceback
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch

REPO = Path("/kaggle/working/pa1-beyond-iid")
RUN_ROOT = Path("/kaggle/working/supplemental_overnight")
LOG_ROOT = RUN_ROOT / "logs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a Kaggle GPU before starting the saved run"
print("GPU:", torch.cuda.get_device_name(0))
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

required = [
    REPO / "task2/train_supplemental.py",
    REPO / "task2/evaluate_supplemental.py",
    REPO / "task3/train_supplemental.py",
    REPO / "task3/evaluate_supplemental_sources.py",
    REPO / "task3/evaluate_supplemental_sketch.py",
    REPO / "task4/train_supplemental.py",
    REPO / "task4/extract_supplemental.py",
    REPO / "task4/evaluate_supplemental.py",
]
missing = [str(path.relative_to(REPO)) for path in required if not path.exists()]
assert not missing, f"Push the supplemental implementation first. Missing: {missing}"


In [ ]:
# Download and extract PACS.
data_dir = REPO / "data"
data_dir.mkdir(parents=True, exist_ok=True)
pacs_zip = data_dir / "pacs.zip"

subprocess.run([
    sys.executable, "-m", "gdown",
    "1m4X4fROCCXMO0lRLrr6Zz9Vb3974NWhE",
    "-O", str(pacs_zip),
], check=True)

assert zipfile.is_zipfile(pacs_zip), f"Downloaded PACS file is not a ZIP: {pacs_zip}"
with zipfile.ZipFile(pacs_zip) as archive:
    archive.extractall(data_dir)

domains = ("photo", "art_painting", "cartoon", "sketch")
matches = [
    path
    for base in (Path("/kaggle/input"), Path("/kaggle/working"))
    for path in base.rglob("images")
    if path.is_dir() and all((path / domain).is_dir() for domain in domains)
]
matches = sorted(set(path.resolve() for path in matches))
print("Possible PACS image roots:", matches)
assert len(matches) == 1, "Expected exactly one PACS images directory"
PACS_ROOT = str(matches[0])
print("PACS_ROOT =", PACS_ROOT)


In [ ]:
# Restore the exact baseline checkpoints and arrange the attached CIFAR files.
checkpoint_input = Path("/kaggle/input/datasets/ahmadsarfraz345/task2-4-checkpoints")
cifar10_input = Path("/kaggle/input/datasets/ahmadsarfraz345/local-cifar")
cifar100_input = Path("/kaggle/input/datasets/ahmadsarfraz345/local-cifar100")

for path in (checkpoint_input, cifar10_input, cifar100_input):
    assert path.exists(), f"Required attached Kaggle dataset path is missing: {path}"


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def find_by_hash(root, names, expected_hash):
    candidates = []
    for name in names:
        candidates.extend(root.rglob(name))
    print("Checkpoint candidates:", candidates)
    for candidate in candidates:
        if candidate.is_file() and sha256(candidate) == expected_hash:
            return candidate
    found = {str(path): sha256(path) for path in candidates if path.is_file()}
    raise FileNotFoundError(
        f"No checkpoint named {names} has expected SHA-256 {expected_hash}. Found: {found}")


task2_config = json.loads((REPO / "task2/configs/supplemental.json").read_text())
source_checkpoint = find_by_hash(
    checkpoint_input, ("source_only.pt",), task2_config["source_only_checkpoint_sha256"])
source_destination = REPO / "task2/checkpoints/source_only.pt"
source_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_checkpoint, source_destination)

import yaml
task4_config = yaml.safe_load((REPO / "task4/configs/proser_clip5.yaml").read_text())
vanilla_checkpoint = find_by_hash(
    checkpoint_input, ("vanilla.pt", "vanila.pt"), task4_config["vanilla_checkpoint_sha256"])
vanilla_destination = REPO / "task4/checkpoints/vanilla.pt"
vanilla_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(vanilla_checkpoint, vanilla_destination)


def find_cifar10_root(root):
    candidates = [path.parent for path in root.rglob("data_batch_1")
                  if (path.parent / "data_batch_5").is_file()
                  and (path.parent / "test_batch").is_file()]
    assert len(candidates) == 1, f"Expected one CIFAR-10 Python directory, found {candidates}"
    return candidates[0]


def find_cifar100_root(root):
    candidates = [path.parent for path in root.rglob("meta")
                  if (path.parent / "train").is_file()
                  and (path.parent / "test").is_file()]
    assert len(candidates) == 1, f"Expected one CIFAR-100 Python directory, found {candidates}"
    return candidates[0]


CIFAR_ROOT = Path("/kaggle/working/cifar-data")
destination10 = CIFAR_ROOT / "cifar-10-batches-py"
destination100 = CIFAR_ROOT / "cifar-100-python"
shutil.copytree(find_cifar10_root(cifar10_input), destination10, dirs_exist_ok=True)
shutil.copytree(find_cifar100_root(cifar100_input), destination100, dirs_exist_ok=True)

print("Restored source checkpoint:", source_destination, sha256(source_destination))
print("Restored Vanilla checkpoint:", vanilla_destination, sha256(vanilla_destination))
print("CIFAR_ROOT =", CIFAR_ROOT)


In [ ]:
# Execution, validation-only collapse decisions, status recording, and archiving helpers.
STATUS = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "repository_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    "gpu": torch.cuda.get_device_name(0),
    "pacs_root": PACS_ROOT,
    "cifar_root": str(CIFAR_ROOT),
    "tasks": {},
}


def save_status():
    STATUS["updated_utc"] = datetime.now(timezone.utc).isoformat()
    (RUN_ROOT / "status.json").write_text(json.dumps(STATUS, indent=2), encoding="utf-8")


def run_module(label, module, *arguments):
    command = [sys.executable, "-u", "-m", module, *map(str, arguments)]
    log_path = LOG_ROOT / f"{label}.log"
    print("\nRUNNING:", " ".join(command))
    started = time.time()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"Finished {label} in {(time.time() - started) / 60:.1f} minutes")


def baseline_source_f1():
    path = REPO / "task2/results/final_metrics.json"
    if path.exists():
        payload = json.loads(path.read_text(encoding="utf-8"))
        value = payload["methods"]["source_only"]["source_val_mean_macro_f1"]
        return float(value)

    # Robust fallback when the small original result file was not committed.
    from shared.pacs_protocol import load_split
    from task2.models import Classifier
    from task2.train import SPLIT, validate
    device = torch.device("cuda")
    checkpoint = torch.load(source_destination, map_location=device, weights_only=True)
    model = Classifier(pretrained=False).to(device)
    model.load_state_dict(checkpoint["model"])
    _, value = validate(model, Path(PACS_ROOT), load_split(SPLIT), device, 1)
    del model
    torch.cuda.empty_cache()
    return float(value)


BASELINE_SOURCE_F1 = baseline_source_f1()
print("Fixed source-only validation macro-F1:", BASELINE_SOURCE_F1)


def assess_training_record(path):
    record = json.loads(Path(path).read_text(encoding="utf-8"))
    best_row = max(record["history"], key=lambda row: row["source_val_mean_macro_f1"])
    histogram = best_row["source_val_prediction_histogram"]
    largest_fraction = max(histogram) / sum(histogram)
    numeric_values = [
        value for row in record["history"] for value in row.values()
        if isinstance(value, (int, float)) and not isinstance(value, bool)
    ]
    finite = all(math.isfinite(value) for value in numeric_values)
    reasons = []
    if not finite:
        reasons.append("non-finite recorded metric")
    if largest_fraction > 0.90:
        reasons.append(f"largest predicted-class fraction {largest_fraction:.4f} > 0.90")
    if best_row["source_val_mean_macro_f1"] < BASELINE_SOURCE_F1 - 0.05:
        reasons.append(
            f"best source F1 {best_row['source_val_mean_macro_f1']:.4f} is more than "
            f"0.05 below baseline {BASELINE_SOURCE_F1:.4f}")
    return {
        "collapsed": bool(reasons),
        "reasons": reasons,
        "best_epoch": best_row["epoch"],
        "best_source_val_mean_macro_f1": best_row["source_val_mean_macro_f1"],
        "largest_predicted_class_fraction": largest_fraction,
        "finite_recorded_metrics": finite,
    }


def archive_task(task_name):
    source = REPO / task_name / "supplemental"
    output = Path("/kaggle/working") / f"{task_name}_supplemental_artifacts.zip"
    if not source.exists():
        return None
    with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in source.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(REPO))
        for path in LOG_ROOT.glob(f"{task_name}_*.log"):
            archive.write(path, Path("supplemental_overnight/logs") / path.name)
        if (RUN_ROOT / "status.json").exists():
            archive.write(RUN_ROOT / "status.json", "supplemental_overnight/status.json")
    print("Saved", output, f"({output.stat().st_size / 1024**2:.1f} MiB)")
    return output


def guarded(task_name, function):
    STATUS["tasks"][task_name] = {"status": "running", "started_utc": datetime.now(timezone.utc).isoformat()}
    save_status()
    try:
        details = function()
        STATUS["tasks"][task_name].update({"status": "complete", "details": details})
    except Exception as error:
        failure = traceback.format_exc()
        print(failure)
        (LOG_ROOT / f"{task_name}_failure.log").write_text(failure, encoding="utf-8")
        STATUS["tasks"][task_name].update({
            "status": "failed", "error": repr(error), "traceback": failure})
    finally:
        STATUS["tasks"][task_name]["finished_utc"] = datetime.now(timezone.utc).isoformat()
        save_status()
        archive_task(task_name)


In [ ]:
# Task 2: clipped DAN lambda=1, DANN, CDAN; normalized DAN fallback if required.
def run_task2():
    run_module("task2_train_clip_all", "task2.train_supplemental",
               "--data-root", PACS_ROOT, "--method", "clip_all")

    results = REPO / "task2/supplemental/results"
    assessments = {
        name: assess_training_record(results / f"{name}_train.json")
        for name in ("dan_clip5", "dann_clip5", "cdan_clip5")
    }
    variants = ["dan_clip5", "dann_clip5", "cdan_clip5"]

    if assessments["dan_clip5"]["collapsed"]:
        run_module("task2_train_dan_norm_fallback", "task2.train_supplemental",
                   "--data-root", PACS_ROOT, "--method", "fallback")
        assessments["dan_norm_clip5"] = assess_training_record(
            results / "dan_norm_clip5_train.json")
        variants.append("dan_norm_clip5")

    # Written before any supplemental command opens Sketch labels.
    selection = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "target_labels_opened": False,
        "baseline_source_val_macro_f1": BASELINE_SOURCE_F1,
        "collapse_rule": "finite metrics, largest class <= 0.90, source F1 within 0.05 of ERM",
        "assessments": assessments,
        "frozen_variants": variants,
    }
    (results / "selection_manifest.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
    run_module("task2_final_evaluation", "task2.evaluate_supplemental",
               "--data-root", PACS_ROOT, "--methods", ",".join(variants))
    return selection


guarded("task2", run_task2)


In [ ]:
# Task 3: clipped DAN-DG lambda=1; normalized fallback if required.
def run_task3():
    run_module("task3_train_dan_dg_clip5", "task3.train_supplemental",
               "--data-root", PACS_ROOT, "--method", "dan_dg_clip5")

    results = REPO / "task3/supplemental/results"
    assessments = {
        "dan_dg_clip5": assess_training_record(results / "dan_dg_clip5_train.json")
    }
    variants = ["dan_dg_clip5"]

    if assessments["dan_dg_clip5"]["collapsed"]:
        run_module("task3_train_dan_dg_norm_fallback", "task3.train_supplemental",
                   "--data-root", PACS_ROOT, "--method", "dan_dg_norm_clip5")
        assessments["dan_dg_norm_clip5"] = assess_training_record(
            results / "dan_dg_norm_clip5_train.json")
        variants.append("dan_dg_norm_clip5")

    # Written before source diagnostics and before Sketch is opened.
    selection = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "sketch_loaded": False,
        "baseline_source_val_macro_f1": BASELINE_SOURCE_F1,
        "collapse_rule": "finite metrics, largest class <= 0.90, source F1 within 0.05 of ERM",
        "assessments": assessments,
        "frozen_variants": variants,
    }
    (results / "selection_manifest.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
    run_module("task3_source_diagnostics", "task3.evaluate_supplemental_sources",
               "--data-root", PACS_ROOT, "--methods", ",".join(variants))
    run_module("task3_final_sketch_evaluation", "task3.evaluate_supplemental_sketch",
               "--data-root", PACS_ROOT)
    return selection


guarded("task3", run_task3)


In [ ]:
# Task 4: clipped PROSER, then fixed-checkpoint CIFAR-100 evaluation.
def run_task4():
    run_module("task4_train_proser_clip5", "task4.train_supplemental",
               "--data-root", CIFAR_ROOT, "--no-download")

    results = REPO / "task4/supplemental/results"
    record = json.loads((results / "proser_clip5_train.json").read_text(encoding="utf-8"))
    best = max(record["history"], key=lambda row: row["validation_accuracy"])
    selection = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "cifar100_loaded": False,
        "frozen_variant": "proser_clip5",
        "selected_epoch": best["epoch"],
        "best_validation_accuracy": best["validation_accuracy"],
        "best_epoch_dummy_win_rate": best["validation_dummy_win_rate"],
        "final_validation_accuracy": record["history"][-1]["validation_accuracy"],
        "checkpoint_sha256": record["checkpoint_sha256"],
    }
    (results / "selection_manifest.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")

    run_module("task4_extract_fixed_outputs", "task4.extract_supplemental",
               "--data-root", CIFAR_ROOT, "--no-download")
    run_module("task4_final_evaluation", "task4.evaluate_supplemental")
    return selection


guarded("task4", run_task4)


In [ ]:
# Create a combined archive, inventory, and concise completion report.
STATUS["finished_utc"] = datetime.now(timezone.utc).isoformat()
save_status()

inventory = []
for task_name in ("task2", "task3", "task4"):
    results = REPO / task_name / "supplemental/results"
    if results.exists():
        inventory.extend(str(path.relative_to(REPO)) for path in results.rglob("*") if path.is_file())
(RUN_ROOT / "result_inventory.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")

combined = Path("/kaggle/working/all_supplemental_artifacts.zip")
with zipfile.ZipFile(combined, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for task_name in ("task2", "task3", "task4"):
        source = REPO / task_name / "supplemental"
        if source.exists():
            for path in source.rglob("*"):
                if path.is_file():
                    archive.write(path, path.relative_to(REPO))
    for path in RUN_ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(Path("/kaggle/working")))

print("\nFINAL STATUS")
print(json.dumps(STATUS, indent=2))
print("\nDOWNLOAD THESE FILES FROM THE SAVED VERSION OUTPUT:")
for path in sorted(Path("/kaggle/working").glob("*_supplemental_artifacts.zip")):
    print(f"{path}  ({path.stat().st_size / 1024**2:.1f} MiB)")
print(f"{combined}  ({combined.stat().st_size / 1024**2:.1f} MiB)")

failed = [name for name, result in STATUS["tasks"].items() if result["status"] != "complete"]
if failed:
    print("\nSome independent tasks failed:", failed)
    print("Their tracebacks are included under supplemental_overnight/logs in the combined ZIP.")
else:
    print("\nAll supplemental tasks completed successfully.")
